In [1]:
!pip install -q -U google-genai

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# !ls /content/drive/MyDrive/predictions/
# !cp /content/drive/MyDrive/predictions/bigscience_mt0-large_notrain_predictions.json .
# !cp /content/drive/MyDrive/predictions/facebook_nllb-200-3.3B_notrain_predictions.json .
# !cp /content/drive/MyDrive/predictions/google_byt5-base_notrain_predictions.json .

# !cp /content/drive/MyDrive/predictions/mt0_large_3_kernel_temperature_sample_single_edit_predictions.json .
# !cp /content/drive/MyDrive/predictions/nllb_3.3b_3_kernel_temperature_sample_single_edit.json .
# !cp /content/drive/MyDrive/predictions/byt5_base_3_kernel_temperature_sampling_single_edit_predictions.json .

!cp /content/drive/MyDrive/predictions/zipped_predictions.json .


In [2]:
# Zip All the prediction files and assign IDs
import os
TARGETS = {
    'mt0-base': 'bigscience_mt0-large_notrain_predictions.json',
    'mt0-fine-tuned': 'mt0_large_3_kernel_temperature_sample_single_edit_predictions.json',
    'nllb-base': 'facebook_nllb-200-3.3B_notrain_predictions.json',
    'nllb-fine-tuned': 'nllb_3.3b_3_kernel_temperature_sample_single_edit.json',
    'byt5-base': 'google_byt5-base_notrain_predictions.json',
    'byt5-fine-tuned': 'byt5_base_3_kernel_temperature_sampling_single_edit_predictions.json',
}

import json

def zip_predictions(targets, output_filepath):
    """
    Reads multiple prediction JSON files and zips them into a single structured JSON.
    Assumes all files have the exact same length and order of sentences.
    """
    loaded_data = {}

    # 1. Load all prediction files into memory
    for model_name, filepath in targets.items():
        try:
            with open(os.path.join('/content/', filepath), 'r', encoding='utf-8') as f:
                loaded_data[model_name] = json.load(f)
        except FileNotFoundError:
            print(f"Error: Could not find file '{filepath}' for model '{model_name}'.")
            return

    # 2. Establish a reference model to get the base text and length
    reference_model = list(targets.keys())[0]
    num_items = len(loaded_data[reference_model])

    # Sanity check: Ensure all loaded lists are the same length
    for model_name, data in loaded_data.items():
        if len(data) != num_items:
            raise ValueError(
                f"Length mismatch: '{model_name}' has {len(data)} items, "
                f"but reference model '{reference_model}' has {num_items} items."
            )

    zipped_output = {}

    # 3. Iterate through indices and build the zipped item payload
    for i in range(num_items):
        item_id = f"item-{i + 1}"
        reference_item = loaded_data[reference_model][i]

        # Extract the shared ground-truth keys from the reference model
        merged_item = {
            "incorrect": reference_item.get("incorrect"),
            "correct": reference_item.get("correct")
        }

        # Dynamically append the prediction for each model defined in TARGETS
        for model_name in targets:
            # We use .get() to avoid KeyErrors if 'predicted' is occasionally missing
            predicted_text = loaded_data[model_name][i].get("predicted", None)
            merged_item[f"prediction_{model_name}"] = predicted_text

        # Assign to our main dictionary with the generated item ID
        zipped_output[item_id] = merged_item

    # 4. Save the unified dictionary to the output file
    with open(output_filepath, 'w', encoding='utf-8') as f:
        # ensure_ascii=False is required to keep the Urdu characters readable
        json.dump(zipped_output, f, ensure_ascii=False, indent=2)

    print(f"Successfully zipped {num_items} items into '{output_filepath}'.")

if __name__ == "__main__":
    # Execute the function
    zip_predictions(TARGETS, 'zipped_predictions.json')

Error: Could not find file 'bigscience_mt0-large_notrain_predictions.json' for model 'mt0-base'.


In [ ]:
#@title Imports, Constants and Env Variables
import json
import os
import time
import copy
from typing import Callable, List, Dict, Any, Tuple, Optional

# Optional imports for clients - make sure the libs are installed in your runtime
try:
    from google import genai
    from google.genai import types
except Exception:
    genai = None
    types = None

try:
    import openai
except Exception:
    openai = None

# -----------------------
# Config / defaults
# -----------------------
MAX_RETRIES = 4
INITIAL_DELAY = 2  # seconds
DEFAULT_BATCH = 8
DEFAULT_SAVE_EVERY = 50


In [4]:
# -----------------------
# Prompt function contract
# -----------------------
# A prompt_generation function should accept (incorrect_text, predicted_text)
# and return a string prompt for a single item. It should NOT include batch-level
# framing (the batch runner will do that). Additionally it should return
# a small "schema" description object with expected types for validation.
#
# We'll provide a helper wrapper that accepts a prompt_fn and returns
# (single_item_template, schema). Example below shows how to adapt your fluency prompt.


def grammaticality_prompt_fn(incorrect_text, predicted_texts_dict):
   targets_str = "\n".join(
       f"{text}"
       for i, text in enumerate(predicted_texts_dict.values())
   )

   output_format = "[\n" + ",\n".join(
       f'  {{\n    {key}: {text},\n    "score": integer between 1 and 5,\n    "reasoning": "brief explanation (1-2 sentences)"\n  }}'
       for key, text in predicted_texts_dict.items()
   ) + "\n]"

   return f"""The goal of this task is to rank the presented targets based on the quality of the sentences.
The context is the original sentence written by an Urdu learner.
The targets are predicted versions of that sentence by multiple models.
After reading the predicted sentence, please assign a score from a minimum of 1 point to a maximum of 5 points to each target based on the quality of the sentence (note that you can assign the same score multiple times).

Your task is to rate **ONLY the grammaticality** of the target sentence.

Ignore:
- How naturally it flows (fluency), as long as it is grammatically valid.
- Whether it perfectly preserves the original meaning (score strictly on the target's grammar).

Grammaticality scoring scale:
1 — Completely ungrammatical with severe structural errors.
2 — Multiple major grammatical errors that impede understanding.
3 — Noticeable grammatical errors, but the core structure is intact.
4 — Mostly grammatical with only minor technical errors.
5 — Perfect grammatical structure with no errors.


# original sentence
{incorrect_text}


# targets
{targets_str}


# output format
Return this JSON array filling in the scores and reasoning values for each predicted text:
{output_format}
"""

def meaning_preservation_prompt_fn(incorrect_text, predicted_texts_dict):
   targets_str = "\n".join(
       f"{text}"
       for i, text in enumerate(predicted_texts_dict.values())
   )

   output_format = "[\n" + ",\n".join(
       f'  {{\n    {key}: {text},\n    "score": integer between 1 and 5,\n    "reasoning": "brief explanation (1-2 sentences)"\n  }}'
       for key, text in predicted_texts_dict.items()
   ) + "\n]"

   return f"""The goal of this task is to rank the presented targets based on the quality of the sentences.
The context is the original sentence written by an Urdu learner.
The targets are predicted versions of that sentence by multiple models.
After reading the predicted sentence, please assign a score from a minimum of 1 point to a maximum of 5 points to each target based on the quality of the sentence (note that you can assign the same score multiple times).

Your task is to rate **ONLY how well the target preserves the original meaning** of the context.

Ignore:
- Minor grammatical or fluency issues in the target sentence, as long as the meaning is conveyed.

Meaning Preservation scoring scale:
1 — Completely changes or loses the original meaning.
2 — Significant portions of the original meaning are lost or altered.
3 — Captures the general idea, but misses important nuances or details.
4 — Preserves almost all meaning with very minor deviations.
5 — Perfectly preserves the original intent and meaning of the learner's sentence.


# original sentence
{incorrect_text}


# targets
{targets_str}


# output format
Return this JSON array filling in the scores and reasoning values for each predicted text:
{output_format}
"""

def fluency_prompt_fn(incorrect_text, predicted_texts_dict):
   targets_str = "\n".join(
       f"{text}"
       for i, text in enumerate(predicted_texts_dict.values())
   )

   output_format = "[\n" + ",\n".join(
       f'  {{\n    {key}: {text},\n    "score": integer between 1 and 5,\n    "reasoning": "brief explanation (1-2 sentences)"\n  }}'
       for key, text in predicted_texts_dict.items()
   ) + "\n]"

   return f"""The goal of this task is to rank the presented targets based on the quality of the sentences.
The context is the original sentence written by an Urdu learner.
The targets are predicted versions of that sentence by multiple models.
After reading the predicted sentence, please assign a score from a minimum of 1 point to a maximum of 5 points to each target based on the quality of the sentence (note that you can assign the same score multiple times).

Your task is to rate **ONLY the fluency** of the target sentence.

Ignore:
- Minor spelling mistakes
- Whether the meaning perfectly matches the learner sentence
- Different wording choices if the sentence still sounds natural


Fluency scoring scale:
1 — Completely unnatural or broken Urdu  
2 — Very awkward and difficult to read naturally  
3 — Understandable but noticeably unnatural  
4 — Mostly natural with minor awkwardness  
5 — Completely natural and fluent Urdu


# original sentence
{incorrect_text}


# targets
{targets_str}


# output format
Return this JSON array filling in the scores and reasoning values for each predicted text:
{output_format}
"""


def fluency_old_prompt_fn(incorrect_text, predicted_texts_dict):
   targets_str = "\n".join(
       f"{text}"
       for i, text in enumerate(predicted_texts_dict.values())
   )

   output_format = "[\n" + ",\n".join(
       f'  {{\n    {key}: {text},\n    "score": integer between 1 and 5,\n    "reasoning": "brief explanation (1-2 sentences)"\n  }}'
       for key, text in predicted_texts_dict.items()
   ) + "\n]"

   return f"""You are evaluating the **fluency** of an Urdu sentence correction.

The context is a sentence written by an Urdu learner.
The targets are predicted versions of that sentence by multiple models.


Your task is to rate **ONLY the fluency** of the target sentence.


Fluency means:
- The sentence sounds natural to a native Urdu speaker
- Words flow smoothly and form a coherent sentence
- Grammar and structure support natural reading
- The sentence is not awkward, broken, or unnatural


Ignore:
- Minor spelling mistakes
- Whether the meaning perfectly matches the learner sentence
- Different wording choices if the sentence still sounds natural


Fluency scoring scale:


1 — Completely unnatural or broken Urdu  
2 — Very awkward and difficult to read naturally  
3 — Understandable but noticeably unnatural  
4 — Mostly natural with minor awkwardness  
5 — Completely natural and fluent Urdu


# original sentence
{incorrect_text}


# targets
{targets_str}


# output format
Return this JSON array filling in the scores and reasoning values for each predicted text:
{output_format}
"""





In [13]:
#@title evaluation_pipeline.py
# -----------------------
# Utility I/O functions
# -----------------------
def load_json(path: str) -> Optional[Any]:
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(path: str, data: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"[I/O] Saved: {path} ({len(data) if isinstance(data, list) else 'n/a'} items)")


def normalize_batch_response(resp_json: Any) -> List[Dict[str, Any]]:
    """
    Accept several plausible JSON shapes and return a list of item objects:
    - [{"score":..,"reasoning":..}, ...]
    - {"results": [ ... ]}
    - {"items": [ ... ]}
    """
    if isinstance(resp_json, list):
        return resp_json
    if isinstance(resp_json, dict):
        for k in ("results", "items", "answers"):
            if k in resp_json and isinstance(resp_json[k], list):
                return resp_json[k]
    raise ValueError("Unrecognized batch response format")

import json
import time
import uuid
from typing import List, Callable, Dict, Any, Optional

from google import genai
from google.genai import types


def run_gemini_batch_api_evaluation(
    zipped_sentences: List[str],
    prompt_function: Callable[[str, str], str],
    model_name: str = "gemini-3-flash-preview",
    temperature: float = 0.0,
    poll_interval: int = 120,  # seconds
    max_wait_time: int = 60 * 60 * 24,  # 24 hours
):
    """
    Gemini Batch API (JSONL-based, async execution).
    Returns results in same order as input.
    """

    client = genai.Client()

    # -----------------------
    # 1. Create JSONL file
    # -----------------------
    file_id = str(uuid.uuid4())
    jsonl_path = f"/tmp/gemini_batch_{file_id}.jsonl"

    requests = []

    for i, zip_item_key in enumerate(zipped_sentences):
        zip_item = zipped_sentences[zip_item_key]
        targets = {k: zip_item[k] for k in zip_item if k.startswith("prediction_")}
        o = zip_item["incorrect"]
        prompt = prompt_function(o, targets)

        request_obj = {
            "key": zip_item_key,
            "request": {
                "contents": [
                    {
                        "parts": [{"text": prompt}]
                    }
                ],
                "generation_config": {
                    "temperature": temperature,
                    "response_mime_type": "application/json"
                }
            }
        }

        requests.append(request_obj)

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for r in requests:
            f.write(json.dumps(r) + "\n")

    print(f"[Batch] JSONL created at {jsonl_path}")

    # -----------------------
    # 2. Upload file
    # -----------------------
    uploaded_file = client.files.upload(
        file=jsonl_path,
        config=types.UploadFileConfig(
            display_name=f"gemini-batch-{file_id}",
            mime_type="application/jsonl"
        )
    )

    print(f"[Batch] Uploaded file: {uploaded_file.name}")

    # -----------------------
    # 3. Create batch job
    # -----------------------
    batch_job = client.batches.create(
        model=model_name,
        src=uploaded_file.name,
        config={
            'display_name': f"gemini-batch-{file_id}",
        },
    )

    print(f"[Batch] Job created: {batch_job.name}")

    # -----------------------
    # 4. Poll until completion
    # -----------------------
    start_time = time.time()

    while True:
        job = client.batches.get(name=batch_job.name)

        state = job.state.name
        print(f"[Batch] State: {state}")

        if state == "JOB_STATE_SUCCEEDED":
            break

        if state in ["JOB_STATE_FAILED", "JOB_STATE_CANCELLED"]:
            raise RuntimeError(f"Batch failed with state: {state}")

        if time.time() - start_time > max_wait_time:
            raise TimeoutError("Batch job exceeded max wait time")

        time.sleep(poll_interval)

    # -----------------------
    # 5. Download results file
    # -----------------------
    result_file = job.dest.file_name
    print(f"[Batch] Result file: {result_file}")

    raw_bytes = client.files.download(file=result_file)
    raw_text = raw_bytes.decode("utf-8")

    # -----------------------
    # 6. Parse JSONL output
    # -----------------------
    results_map = {}

    for line in raw_text.splitlines():
        obj = json.loads(line)

        key = obj.get("key")

        # response structure
        response_text = obj["response"]["candidates"][0]["content"]["parts"][0]["text"]

        results_map[key] = json.loads(response_text)

    # -----------------------
    # 7. Return in original order
    # -----------------------
    ordered_results = []
    for i in range(len(zipped_sentences)):
        ordered_results.append(results_map[f"item-{i}"])

    return ordered_results

def run_gemini_evaluation_seq(
    zipped_sentences: Dict[str, Any],
    prompt_function: Callable,
    model_name: str = "gemini-3-flash-preview",
    max_retries: int = MAX_RETRIES,
    initial_delay: float = INITIAL_DELAY,
) -> Dict[str, Any]:
    """
    Single-item Gemini evaluation (no prompt batching).
    
    - 1 request per item
    - retries per item
    - never crashes whole run
    - returns dict: key -> result (or None on failure)
    """

    if genai is None:
        raise RuntimeError("google.genai client not available.")

    client = genai.Client()
    results = {}

    for idx, (key, item) in enumerate(zipped_sentences.items()):
        print(f"[Gemini] Processing {key} ({idx+1}/{len(zipped_sentences)})")

        # -----------------------
        # Build prompt inputs
        # -----------------------
        incorrect = item.get("incorrect")
        targets = {k: item[k] for k in item if k.startswith("prediction_")}

        if not incorrect or not targets:
            print(f"[Gemini][{key}] Missing data, skipping")
            results[key] = None
            continue

        prompt = prompt_function(incorrect, targets)

        # -----------------------
        # Retry loop (per item)
        # -----------------------
        delay = initial_delay
        success = False

        for attempt in range(1, max_retries + 1):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        response_mime_type="application/json",
                        temperature=0.0,
                    ),
                )

                text = getattr(response, "text", None)
                if text is None:
                    text = json.dumps(response) if not isinstance(response, str) else str(response)

                parsed = json.loads(text)

                # Expecting list output (multiple targets)
                if not isinstance(parsed, list):
                    raise ValueError("Expected list output for multiple targets")

                # Optional: basic validation per item
                for i, obj in enumerate(parsed):
                    if not isinstance(obj, dict):
                        raise ValueError(f"Item {i} is not an object")
                    if "score" not in obj or "reasoning" not in obj:
                        raise ValueError(f"Missing keys in item {i}")

                results[key] = parsed
                success = True
                break

            except Exception as e:
                print(f"[Gemini][{key}] attempt {attempt} failed: {e}")

                if attempt < max_retries:
                    time.sleep(delay)
                    delay *= 2
                else:
                    print(f"[Gemini][{key}] FAILED permanently")
                    results[key] = None  # no crash

        if not success and key not in results:
            results[key] = None

    return results

def run_evaluation(
    eval_fn: Callable[[Dict[str, Any], Callable], Dict[str, Any]],
    input_file: str,
    output_file: str,
    prompt_fn: Callable,
    save_every: int = 10,  # <-- NEW: save every k items
):
    """
    Incremental orchestrator:
    - loads dataset
    - resumes from partial output
    - processes only missing keys
    - saves every `save_every` items
    """

    # -----------------------
    # 1. Load input
    # -----------------------
    data = load_json(input_file)
    if data is None:
        raise FileNotFoundError(f"Input file not found: {input_file}")

    print(f"[Main] Loaded {len(data)} items")

    # -----------------------
    # 2. Load existing output (resume)
    # -----------------------
    existing = load_json(output_file) or {}
    print(f"[Main] Loaded existing results: {len(existing)} items")

    outputs = existing.copy()

    # -----------------------
    # 3. Determine remaining items
    # -----------------------
    remaining = {
        k: v for k, v in data.items()
        if k not in outputs or "gemini_result" not in outputs[k] or outputs[k]["gemini_result"] is None
    }

    print(f"[Main] Remaining to process: {len(remaining)}")

    if not remaining:
        print("[Main] Nothing to do — all items already processed")
        return

    # -----------------------
    # 4. Process one-by-one (resume-safe)
    # -----------------------
    processed_count = 0

    for key, item in remaining.items():
        print(f"[Main] Processing {key}")

        try:
            result = eval_fn(
                zipped_sentences={key: item},
                prompt_function=prompt_fn,
            ).get(key)

        except Exception as e:
            print(f"[Error] {key} failed: {e}")
            result = None

        # Merge into output
        merged = item.copy()
        merged["gemini_result"] = result
        outputs[key] = merged

        processed_count += 1

        # -----------------------
        # 5. Save progress
        # -----------------------
        if processed_count % save_every == 0:
            save_json(output_file, outputs)
            print(f"[Main] Progress saved ({processed_count} new items)")

    # Final save
    save_json(output_file, outputs)
    print(f"[Main] Done. Total saved: {len(outputs)}")

In [ ]:
# Grammatically

run_evaluation(
    eval_fn=run_gemini_evaluation_seq,
    input_file='../tmp/zipped_predictions.json',
    output_file='../tmp/grammatically_output.json',
    prompt_fn=grammaticality_prompt_fn,
)

# Fluency
run_evaluation(
    eval_fn=run_gemini_evaluation_seq,
    input_file='../tmp/zipped_predictions.json',
    output_file='../tmp/fluency_output.json',
    prompt_fn=fluency_prompt_fn,
)

# Meaning Preservation
run_evaluation(
    eval_fn=run_gemini_evaluation_seq,
    input_file='../tmp/zipped_predictions.json',
    output_file='../tmp/meaning_preservation_output.json',
    prompt_fn=meaning_preservation_prompt_fn,
)

[Main] Loaded 1613 items
[Main] Loaded existing results: 0 items
[Main] Remaining to process: 1613
[Main] Processing item-1
[Gemini] Processing item-1 (1/1)
[Main] Processing item-2
[Gemini] Processing item-2 (1/1)
[Main] Processing item-3
[Gemini] Processing item-3 (1/1)
[Main] Processing item-4
[Gemini] Processing item-4 (1/1)
[Main] Processing item-5
[Gemini] Processing item-5 (1/1)
[Main] Processing item-6
[Gemini] Processing item-6 (1/1)
[Main] Processing item-7
[Gemini] Processing item-7 (1/1)
[Main] Processing item-8
[Gemini] Processing item-8 (1/1)
[Gemini][item-8] attempt 1 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
[Main] Processing item-9
[Gemini] Processing item-9 (1/1)
[Main] Processing item-10
[Gemini] Processing item-10 (1/1)
[I/O] Saved: ../tmp/grammatically_output.json (n/a items)
[Main] Progress saved (10 new i

KeyboardInterrupt: 

In [ ]:
!cat /content/mt0_large_3_kernel_temperature_sample_single_edit_predictions_with_scores.json

[
  {
    "incorrect": "کیا مسلمانوں کی عبادت میں کوئی کمی رہ گئی ہے یہ پھر اللہ کی عنایت کا انداز بدل گیا ہے۔",
    "correct": "کیا مسلمانوں کی عبادت میں کوئی کمی رہ گئی ہے یا پھر اللہ کی عنایت کا انداز بدل گیا ہے۔",
    "predicted": "کیا مسلمانوں کی عبادت میں کوئی کمی رہ گئی ہے یہ پھر اللہ کی عنایت کا انداز بدل گیا۔",
    "incorrect_pair": "کیا مسلمانوں کی عبادت میں کوئی کمی رہ گئی ہے یہ پھر اللہ کی عنایت کا انداز بدل گیا ہے۔",
    "correct_pair": "کیا مسلمانوں کی عبادت میں کوئی کمی رہ گئی ہے یا پھر اللہ کی عنایت کا انداز بدل گیا ہے۔",
    "gemini_score": 3,
    "gemini_reasoning": "The use of 'یہ' (this) instead of the conjunction 'یا' (or) is grammatically incorrect and disrupts the natural flow of the sentence."
  },
  {
    "incorrect": "تو شکایت اس بات یہ ہے کہ پروردگار مسلمان تیری عبادت کرتے، ہر دم یاد کرتے ہیں لیکن نوازش انگریز کو ہو رہی ہے۔",
    "correct": "تو شکایت اس بات کی ہے کہ پروردگار مسلمان تیری عبادت کرتے، ہر دم یاد کرتے ہیں لیکن نوازش انگریز کو ہو رہی ہے۔",
    "pre

In [ ]:
!cp mt0_large_3_kernel_temperature_sample_single_edit_predictions_with_scores.json /content/drive/MyDrive/mt0_LLMscores/

In [ ]:
!cp bigscience_mt0-large_notrain_predictions_with_scores.json /content/drive/MyDrive/mt0_LLMscores/bigscience_mt0-large_notrain_predictions_with_scores_nofail.json

In [ ]:
#@title Compare mt0_large_3_kernel_temperature_sample_single_edit_predictions with bigscience_mt0-large_notrain_predictions


import json


def load_scores(filepath):
    """Load gemini scores from a JSON file."""
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    scores = []

    for item in data:
        score = item.get("gemini_score")

        # Keep only valid numeric scores
        if isinstance(score, (int, float)):
            scores.append(score)

    return scores


def compute_average(scores):
    """Compute average safely."""
    if not scores:
        return 0.0
    return sum(scores) / len(scores)


def main():
    file1 = "/content/mt0_large_3_kernel_temperature_sample_single_edit_predictions_with_scores.json"
    file2 = "/content/bigscience_mt0-large_notrain_predictions_with_scores.json"

    scores1 = load_scores(file1)
    scores2 = load_scores(file2)

    avg1 = compute_average(scores1)
    avg2 = compute_average(scores2)

    print(f"File 1: {file1}")
    print(f"  Count: {len(scores1)}")
    print(f"  Average Score: {avg1:.4f}")
    print()

    print(f"File 2: {file2}")
    print(f"  Count: {len(scores2)}")
    print(f"  Average Score: {avg2:.4f}")


if __name__ == "__main__":
    main()


File 1: /content/mt0_large_3_kernel_temperature_sample_single_edit_predictions_with_scores.json
  Count: 1613
  Average Score: 4.1339

File 2: /content/bigscience_mt0-large_notrain_predictions_with_scores.json
  Count: 1613
  Average Score: 4.0099


In [ ]:
!cp  /content/drive/MyDrive/mt0_LLMscores/gold_auto_plus_spell_tags.txt .

In [ ]:
#@title Bucket Logic
import json
from collections import defaultdict


SCORES_FILE = "/content/bigscience_mt0-large_notrain_predictions_with_scores_nofail.json"
ERROR_FILE = "/content/gold_auto_plus_spell_tags.txt"


def load_scores(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    scores = []

    for item in data:
        score = item.get("gemini_score")

        if isinstance(score, (int, float)):
            scores.append(score)
        else:
            scores.append(None)

    return scores


def load_error_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f.readlines()]


def main():

    scores = load_scores(SCORES_FILE)
    errors = load_error_lines(ERROR_FILE)

    if len(scores) != len(errors):
        raise ValueError("Length mismatch between scores and error file")

    category_scores = defaultdict(list)

    for score, err_line in zip(scores, errors):

        if score is None:
            continue

        if not err_line:
            continue

        tags = err_line.split()

        for tag in tags:
            category_scores[tag].append(score)

    print("\nAverage Score by Fine-Grained Category")
    print("=" * 40)
    num_errors = 0
    for cat in sorted(category_scores.keys()):
        vals = category_scores[cat]
        avg = sum(vals) / len(vals)
        num_errors += len(vals)

        print(f"{cat:20s}  count={len(vals):4d}  avg={avg:.4f}")
    print(len(category_scores.keys()), num_errors)


if __name__ == "__main__":
    main()



Average Score by Fine-Grained Category
M:ADJ                 count=  30  avg=3.5000
M:ADP                 count= 156  avg=3.9679
M:ADV                 count=   2  avg=2.5000
M:AUX                 count=  71  avg=3.9296
M:CCONJ               count=  26  avg=3.6154
M:DET                 count=  13  avg=3.4615
M:NOUN                count=  69  avg=3.8116
M:NUM                 count=  15  avg=4.0000
M:PART                count=  41  avg=3.5854
M:PRON                count=  74  avg=3.7973
M:PROPN               count=  10  avg=4.0000
M:PUNCT               count=  19  avg=3.7895
M:SCONJ               count=  11  avg=3.3636
M:VERB                count=  40  avg=3.3250
M:X                   count=   1  avg=5.0000
NONE                  count=  19  avg=3.6842
R:ADJ                 count=  81  avg=3.6914
R:ADJ:INFL            count=  16  avg=3.8125
R:ADP                 count= 159  avg=4.0063
R:ADP:INFL            count= 129  avg=3.9922
R:ADV                 count=  13  avg=3.3077
R:ADV:INFL     

In [ ]:
#!/usr/bin/env python3

import json
from collections import defaultdict
from typing import List, Dict

# ---------- CONFIG ----------
SCORES_FILE = "/content/mt0_large_3_kernel_temperature_sample_single_edit_predictions_with_scores.json"
ERROR_FILE = "gold_auto_plus_spell_tags.txt"
OUTPUT_SUMMARY_JSON = "aggregates_single_file_fientuned.json"  # set to None to skip saving
# ----------------------------


BASE_TO_MACRO = {
    # Verb & Auxiliaries
    "VERB": "Verb & Auxiliaries",
    "AUX":  "Verb & Auxiliaries",

    # Nouns & Pronouns
    "NOUN":  "Nouns & Pronouns",
    "PRON":  "Nouns & Pronouns",
    "PROPN": "Nouns & Pronouns",

    # Adpositions
    "ADP": "Adpositions",

    # Modifiers & Miscellaneous
    "ADJ":   "Modifiers & Miscellaneous",
    "ADV":   "Modifiers & Miscellaneous",
    "DET":   "Modifiers & Miscellaneous",
    "CCONJ": "Modifiers & Miscellaneous",
    "SCONJ": "Modifiers & Miscellaneous",
    "PART":  "Modifiers & Miscellaneous",
    "NUM":   "Modifiers & Miscellaneous",
    "X":     "Modifiers & Miscellaneous",

    # Orthography
    "SPELL": "Orthography",
    "PUNCT": "Orthography",

    "NONE": "NONE",
}


def load_scores(path: str) -> List[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_error_lines(path: str) -> List[str]:
    with open(path, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]


def extract_fine_tags(line: str) -> List[str]:
    if not line or not line.strip():
        return []

    tokens = line.split()
    tags = []

    for tok in tokens:
        parts = tok.split(":", 1)
        if len(parts) == 2:
            tags.append(parts[1])
        else:
            tags.append(tok)

    return tags


def map_tag_to_macro(fine_tag: str) -> str:
    base = fine_tag.split(":", 1)[0]
    return BASE_TO_MACRO.get(base, "OTHER")


def compute_aggregates(scores_data: List[dict], error_lines: List[str]):

    if len(scores_data) != len(error_lines):
        raise ValueError("Length mismatch between scores and errors")

    fine_count_sum = defaultdict(lambda: {"count": 0, "sum": 0.0})
    macro_count_sum = defaultdict(lambda: {"count": 0, "sum": 0.0})

    overall_count = 0
    overall_sum = 0.0

    for item, err_line in zip(scores_data, error_lines):

        score = item.get("gemini_score")

        if score is None:
            continue

        try:
            val = float(score)
        except Exception:
            continue

        # overall stats
        overall_count += 1
        overall_sum += val

        fine_tags = extract_fine_tags(err_line)

        for fine in fine_tags:

            fine_count_sum[fine]["count"] += 1
            fine_count_sum[fine]["sum"] += val

            macro = map_tag_to_macro(fine)
            macro_count_sum[macro]["count"] += 1
            macro_count_sum[macro]["sum"] += val

    # Convert to averages
    def finalize(d):
        out = {}
        for k, cs in d.items():
            cnt = cs["count"]
            s = cs["sum"]
            out[k] = {"count": cnt, "avg": (s / cnt) if cnt else 0.0}
        return out

    fine_result = finalize(fine_count_sum)
    macro_result = finalize(macro_count_sum)

    overall_avg = overall_sum / overall_count if overall_count else 0.0

    overall_result = {
        "count": overall_count,
        "avg": overall_avg
    }

    return fine_result, macro_result, overall_result


def pretty_print(title: str, d: Dict[str, Dict]):
    print("=" * 60)
    print(title)
    print("=" * 60)

    for k in sorted(d.keys()):
        print(f"{k:25s}  count={d[k]['count']:5d}  avg={d[k]['avg']:7.4f}")

    print()


def main():

    scores = load_scores(SCORES_FILE)
    errors = load_error_lines(ERROR_FILE)

    fine_res, macro_res, overall_res = compute_aggregates(scores, errors)

    pretty_print("Fine-grained tag aggregates", fine_res)
    pretty_print("Macro category aggregates", macro_res)

    print("=" * 60)
    print("OVERALL SCORE")
    print("=" * 60)
    print(f"count={overall_res['count']}  avg={overall_res['avg']:.4f}")
    print()

    if OUTPUT_SUMMARY_JSON:
        out = {
            "fine_grained": fine_res,
            "macro": macro_res,
            "overall": overall_res
        }

        with open(OUTPUT_SUMMARY_JSON, "w", encoding="utf-8") as f:
            json.dump(out, f, ensure_ascii=False, indent=2)

        print("Saved aggregates to", OUTPUT_SUMMARY_JSON)


if __name__ == "__main__":
    main()


Fine-grained tag aggregates
ADJ                        count=  131  avg= 3.4962
ADJ:INFL                   count=   16  avg= 4.0625
ADP                        count=  431  avg= 3.8260
ADP:INFL                   count=  129  avg= 4.2326
ADV                        count=   18  avg= 4.2222
ADV:INFL                   count=    1  avg= 3.0000
AUX                        count=  245  avg= 3.5224
AUX:INFL                   count=  290  avg= 4.1207
CCONJ                      count=   80  avg= 3.7625
DET                        count=   31  avg= 3.0323
DET:INFL                   count=   16  avg= 4.0000
NONE                       count=   19  avg= 4.1053
NOUN                       count=  404  avg= 3.6931
NOUN:INFL                  count=   88  avg= 4.1818
NUM                        count=   31  avg= 3.8065
PART                       count=   89  avg= 3.5506
PRON                       count=  245  avg= 3.5429
PRON:INFL                  count=   73  avg= 3.8493
PROPN                      count=   

In [ ]:
# old
